# Final Part  — Out-of-Sample Generalization and Robustness Analysis
## <b> ADRIAN VAZQUEZ </b>
---

In [ ]:
# libs
import plotly.io as pio
#pio.renderers.default = "notebook_connected+png" 

from IPython.display import Image, display
import sys
import os
# Add project root to path
sys.path.append(os.path.abspath(".."))

# libs 
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# RF 
import torch
import torch.nn as nn
import torch.optim as optim

from src.models.twap import twap_schedule
from src.models.vwap import generate_vwap_schedule
from src.models.inventory_from_schedule import inventory_from_schedule
from src.models.almgren_chriss_shedule import almgren_chriss_schedule
from src.analytics.implementation_shortfall import implementation_shortfall
from src.analytics.get_intraday_prices import get_intraday_prices
from src.models.simulate_brownian_motion_price_path import simulate_brownian_price_path, simulate_multiple_price_paths, simulate_execution_prices
from src.models.volume_aware_ac_schedule import volume_aware_ac_schedule

# <b> Data Acquisition  </b> 

In [ ]:
#  Download intraday market data
API_KEY = "API_KEY"

df_spy_5min = get_intraday_prices(
    symbol="SPY",
    api_key=API_KEY,
    interval="5min",
    outputsize="full",
    extended_hours="false",
    adjusted=False
)

# Cleaning

df_spy_5min.index = pd.to_datetime(df_spy_5min.index)

df_spy_5min = (
    df_spy_5min
    .sort_index()
    .loc[~df_spy_5min.index.duplicated()]
)

df_spy_5min = df_spy_5min.between_time("09:30", "15:55")


# Market feature engineering


df_market_state = df_spy_5min.copy()

# Price dynamics
df_market_state["returns"] = df_market_state["Close"].pct_change()

df_market_state["rolling_vol"] = (
    df_market_state["returns"]
    .rolling(window=20)
    .std()
)

df_market_state["momentum"] = (
    df_market_state["Close"]
    - df_market_state["Close"].rolling(window=5).mean()
)

# Liquidity dynamics
df_market_state["rel_volume"] = (
    df_market_state["Volume"]
    / df_market_state["Volume"].rolling(window=20).mean()
)

df_market_state["minute_of_day"] = (
    df_market_state.index.hour * 60
    + df_market_state.index.minute
)

volume_profile = (
    df_market_state
    .groupby("minute_of_day")["Volume"]
    .mean()
)

df_market_state["volume_profile"] = (
    df_market_state["minute_of_day"]
    .map(volume_profile)
)

df_market_state["volume_profile_norm"] = (
    df_market_state["volume_profile"]
    / df_market_state["volume_profile"].mean()
)

# Friction proxy
df_market_state["spread_proxy"] = (
    df_market_state["High"]
    - df_market_state["Low"]
)

df_market_state["spread_proxy_pct"] = (
    df_market_state["spread_proxy"]
    / df_market_state["Close"]
)

# Final RL market state dataset

market_feature_cols = [
    "returns",
    "rolling_vol",
    "momentum",
    "rel_volume",
    "volume_profile_norm",
    "spread_proxy_pct"
]

df_market_state = df_market_state.dropna(
    subset=market_feature_cols
)

print("Market state dataset shape:", df_market_state.shape)
print("Start:", df_market_state.index.min())
print("End:", df_market_state.index.max())

df_market_state[market_feature_cols].head()

In [ ]:
# plot
df_spy_5min = df_market_state.copy()
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df_spy_5min.index,
        y=df_spy_5min["Close"],
        mode="lines",
        name="SPY Close"
    )
)
fig.update_layout(
    title="SPY Intraday Price (5-min)",
    xaxis_title="Time",
    yaxis_title="Price",
    height=450,
    width=900
)
fig.show()
fig.write_image("../results/plots/2_SPY_Intraday_Price_5min.png")
display(Image(filename="../results/plots/2_SPY_Intraday_Price_5min.png"))

# <b> Reusing RL Execution Components. </b> 

To keep this notebook focused on out-of-sample evaluation and robustness analysis, the main classes and functions developed in the previous section are now moved into a reusable Python module.

The logic remains unchanged. The only difference is that the implementation is now packaged inside:

```python

## <b> Calling the classes and functions that  we have been using so far from '05_RL_Execution_agent.py' </b>

They are the exact same functions; the only difference is that now we have  packed them into a .py file.

The Classes and fucntions are: 
 
- StateBuilder() class 
  1. buil_state
- ExecutionEnv() class
  1. get_execution_history
- run_random_policy
- run_twap_policy
- run_vwap_like_policy
- run_volume_aware_ac_policy
- DQN () class
  1. forward 
- ReplayBuffer() class
  1. push 
  2. sample
- select_action
- update_target_network
- train_dqn_step   <- <b> Here, we are goin to train the agent over many days  </b>
- run_dqn_policy


In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))
from src.models.RL_Execution import StateBuilder
from src.models.RL_Execution import ExecutionEnv